#### ENRICH GEO LOCATIONS FROM BRONZE TO SILVER

In [0]:
CATALOG = 'ORDERS_SALES'
SCHEMA = 'SILVER'
TARGET_TABLE = 'GEO_LOCATIONS_ENRICHED'

In [0]:
geo_locations_df = spark.table('ORDERS_SALES.BRONZE.GEO_LOCATIONS')
geo_locations_df.limit(1).display()

In [0]:
from pyspark.sql.functions import *

geo_locations_trans_df = (geo_locations_df.select(
    "geolocation_zip_code_prefix",
    "geolocation_lat",
    "geolocation_lng",
    "geolocation_state",
    "geolocation_city",
    "file_name",
    "file_path",
    current_timestamp().alias('load_timestamp')
)
.withColumn("zip_prefix", col("geolocation_zip_code_prefix").cast("int"))
.withColumn("lat", col("geolocation_lat").cast("double"))
.withColumn("lng", col("geolocation_lng").cast("double"))
.withColumnRenamed("geolocation_state", "state")
.withColumnRenamed("geolocation_city", "city")
.drop("geolocation_lng", "geolocation_lat")
)

geo_locations_selected_df = geo_locations_trans_df.select(
    "zip_prefix",
    "state",
    upper(col("city")).alias("city"),
    "lat",
    "lng",
    "file_path",
    current_timestamp().alias("load_timestamp")
)
display(geo_locations_selected_df.limit(1))

In [0]:
geo_locations_selected_df.createOrReplaceTempView("geo_locations_selected_temp_vw")

#### INCREMENTAL LOAD `ORDERS_SALES.SILVER.GEO_LOCATIONS_ENRICHED`

In [0]:
spark.sql(f"""
MERGE INTO {CATALOG}.{SCHEMA}.{TARGET_TABLE} tgt
USING (
  SELECT *
  FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY zip_prefix
               ORDER BY load_timestamp DESC
           ) rn
    FROM geo_locations_selected_temp_vw
  )
  WHERE rn = 1
) src
ON (src.zip_prefix = tgt.zip_prefix)
WHEN MATCHED THEN
  UPDATE SET
    tgt.state = src.state,
    tgt.city = src.city,
    tgt.lat = src.lat,
    tgt.lng = src.lng
WHEN NOT MATCHED THEN
  INSERT (
    zip_prefix,
    state,
    city,
    lat,
    lng,
    origin_file_path,
    load_timestamp
  )
  VALUES (
    src.zip_prefix,
    src.state,
    src.city,
    src.lat,
    src.lng,
    src.file_path,
    current_timestamp()
  )
""")

In [0]:
dbutils.notebook.exit('SUCCESSFULLY LOADED INTO ORDERS_SALES.SILVER.GEO_LOCATIONS_ENRICHED')